# Import releveant Libraries

In [1]:
!pip install tiktoken

In [2]:
import torch #PyTorch
from torch.utils.data import Dataset, DataLoader #Pytorch Dataset and Dataloader Classes
import urllib.request #so we can read data from a URL to a file
import os #so we can read data from the file
import tiktoken #Tokenization Library

# PyTorch Dataset and DataLoader

In [3]:
#Since PyTorch's Dataset class is an abstract class, we cannot directly create objects from the Dataset class.
#Instead, we need to create a custom class that inherits from the PyTorch's Dataset class.
#In our implementation, we must specifically implement the following functions: __init__, __getitem__, and __len__
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    def __len__(self):
        return self.labels.shape[0]

In [4]:
# Example
X_train = torch.tensor([
[-1.2, 3.1],
[-0.9, 2.9],
[-0.5, 2.6],
[2.3, -1.1],
[2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
[-0.8, 2.8],
[2.6, -1.6],
])
y_test = torch.tensor([0, 1])

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [6]:
#We will convert our data from dataset to dataloader, so we can have more features like batching and shuffling
train_loader = DataLoader(
    dataset=train_ds,      # The ToyDataset instance for training data
    batch_size=2,         # Number of samples per batch
    shuffle=True,         # Shuffle the training data, at the start of each epoch (iteration over the dataset).
    num_workers=0 ,        # Number of background processes
    drop_last = False     # whether to drop the last batch if it's smaller than batch size
)

# Create DataLoader for test dataset
test_loader = DataLoader(
    dataset=test_ds,      # The ToyDataset instance for test data
    batch_size=2,         # Number of samples per batch
    shuffle=False,        # Do not shuffle the test data
    num_workers=0,         # Number of background processes
    drop_last = False     # whether to drop the last batch if it's smaller than batch size
)

### Iterating over a dataloader

In [7]:
#Note that you cannot see the content of a dataloader by just printing it
print(train_loader)

In [8]:
#Instead, you need to iterate over your dataloader
for x,y in train_loader:
    print(x)
    print(y)
    print("---------------------------------")

tensor([[-1.2000,  3.1000],
        [ 2.3000, -1.1000]])
tensor([0, 1])
---------------------------------
tensor([[-0.9000,  2.9000],
        [ 2.7000, -1.5000]])
tensor([0, 1])
---------------------------------
tensor([[-0.5000,  2.6000]])
tensor([0])
---------------------------------


Note that each iteration of the loop corresponds to 1 batch

# Reading Data from URL

In [9]:
#Step 1: Read data from URL to file
if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt") #URL where data exists
    file_path = "the-verdict.txt" #file name where we want to store our data
    urllib.request.urlretrieve(url, file_path) #reading data from URL to file

In [10]:
#Step 2: Read data from file to string
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [11]:
print(raw_text)

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through th

# Tokenization

In [12]:
#Define GPT-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
print(tokenizer.n_vocab) #vocabulary size

50257


The tokenizer has 2 important functions:


*   encode: converts raw string to token IDs
*   decode: converts token IDs to raw_string



In [14]:
print(tokenizer.encode("cat"))

[9246]


In [15]:
print(tokenizer.decode([9246]))

cat


Note that some words can be encoded into more than one token

In [16]:
print(tokenizer.encode("naughty"))

[77, 28496]


In [17]:
print(tokenizer.decode([77]))

n


In [18]:
print(tokenizer.decode([28496]))

aughty


In [19]:
print(tokenizer.decode([77,28496]))

naughty


You can also encode and decode a sentence

In [20]:
print(tokenizer.encode("My cat is naughty"))

[3666, 3797, 318, 45128]


In [21]:
print(tokenizer.decode([3666,3797,318,45128]))

My cat is naughty


In [22]:
print(tokenizer.decode([3666]))

My


In [23]:
print(tokenizer.decode([3797]))

 cat


In [24]:
print(tokenizer.decode([318]))

 is


In [25]:
print(tokenizer.decode([45128]))

 naughty


Now let's encode our data

In [26]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text)) #Should print the vocabulary size (number of tokens)

5145


In [27]:
print(enc_text[:20])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


In [28]:
for token in enc_text[:20]:
    print(tokenizer.decode([token]))

I
 H
AD
 always
 thought
 Jack
 G
is
burn
 rather
 a
 cheap
 genius
--
though
 a
 good
 fellow
 enough
--


# Converting Raw Text into Inputs and Outputs

**We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:**

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [29]:
#First we need to divide 2 hyperparameters: context_size and stride
context_size = 4
stride = 3

In [30]:
#Let's see how to convert raw text into input output pairs
for i in range(0, len(enc_text[:20])-context_size,stride):
  input_chunk = enc_text[i:i + context_size]
  target_chunk = enc_text[i + 1: i + context_size + 1]
  print(input_chunk, "---->", target_chunk)

[40, 367, 2885, 1464] ----> [367, 2885, 1464, 1807]
[1464, 1807, 3619, 402] ----> [1807, 3619, 402, 271]
[402, 271, 10899, 2138] ----> [271, 10899, 2138, 257]
[2138, 257, 7026, 15632] ----> [257, 7026, 15632, 438]
[15632, 438, 2016, 257] ----> [438, 2016, 257, 922]
[257, 922, 5891, 1576] ----> [922, 5891, 1576, 438]


In [31]:
#Now let's create a class called GPTDataset that allows us to input raw_text, tokenizer, context_size, and stride
# and it will create a dataset object with features and labels
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, context_size, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > context_size, "Number of tokenized inputs must at least be equal to context_size+1"

        # Use a sliding window to chunk the book into overlapping sequences of context_size
        for i in range(0, len(token_ids) - context_size, stride):
            input_chunk = token_ids[i:i + context_size]
            target_chunk = token_ids[i + 1: i + context_size + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [32]:
#Now let's create a function that takes txt and other hyperparameters and creates a dataset
#and converts that dataset into a dataloader and returns it
def create_dataloader_v1(txt, batch_size=4, context_size=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, context_size, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [33]:
#Now let's apply this function to create a dataloader to our data
dataloader = create_dataloader_v1(
    raw_text, batch_size=5, context_size=4, stride=1, shuffle=False
)

In [34]:
#Now let's print 1 batch
for inputs, targets in dataloader:
  print("Inputs:",inputs,'\n')
  print("Targets:",targets,'\n')
  break #to stop looping over other batches

Inputs: tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402],
        [1807, 3619,  402,  271]]) 

Targets: tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899]]) 



#Token Embeddings

- The data is already almost ready for an LLM
- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

Let's take a simple example: a vocabulary of 6 tokens only and each token has an embedding of dimension 3

In [35]:
vocab_size = 6
output_dim = 3
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [36]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.2239,  0.1138,  1.2223],
        [ 1.7382,  2.3379,  1.4931],
        [-1.0026, -1.4264, -1.2898],
        [ 0.7766, -0.9684,  1.1988],
        [ 0.2177,  0.8711, -1.4348],
        [-2.0762,  0.1386, -0.4157]], requires_grad=True)


Note that each row corresponds to a token, and the embeddings have been initialized randomly and they will be updated during the training process

In [37]:
#Let's retreive the embedding of token 0
print(embedding_layer(torch.tensor([0])))

tensor([[0.2239, 0.1138, 1.2223]], grad_fn=<EmbeddingBackward0>)


In [38]:
#Let's retreive the embedding of tokens 1,2,5
print(embedding_layer(torch.tensor([1,2,5])))

tensor([[ 1.7382,  2.3379,  1.4931],
        [-1.0026, -1.4264, -1.2898],
        [-2.0762,  0.1386, -0.4157]], grad_fn=<EmbeddingBackward0>)


- An embedding layer is essentially a look-up operation:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

Now, the actual GPT model has a vocab size of 50257 and embeddings of dimension 768

In [39]:
vocab_size = 50257
output_dim = 768
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [40]:
print(embedding_layer.weight.shape)

torch.Size([50257, 768])


In [41]:
print(embedding_layer.weight[:5])

tensor([[-1.7401,  0.6429,  0.2851,  ...,  0.3275, -1.2402, -1.1737],
        [-0.4949,  0.1319,  1.2806,  ...,  0.3255,  1.1318, -0.9417],
        [ 0.6346,  0.9025, -1.1786,  ..., -0.5401,  0.4535, -0.7639],
        [ 0.2313, -0.8486, -0.1645,  ...,  0.9040,  0.1552, -0.0704],
        [-0.2922, -1.1369, -0.6325,  ...,  0.5358,  1.1551,  0.3318]],
       grad_fn=<SliceBackward0>)


# Positional Embeddings

Without positional embeddings, sentences with same words but in different orders will have the same embedding

For example:
The man bit the dog VS The dog bit the man

In [42]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [43]:
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[-0.2437,  0.9585, -0.9505,  ...,  0.2303, -0.2684,  0.7858],
        [-1.8610, -1.9265,  1.0855,  ...,  0.5151,  0.8646, -0.2780],
        [-0.2598,  0.0105,  0.0883,  ..., -2.1603,  0.3655,  1.3001],
        [ 0.7302,  0.2401, -0.9500,  ...,  0.9871, -0.2649,  0.1062]],
       requires_grad=True)


Now let's combine token embeddings with positional embeddings

In [44]:
token_embeddings = embedding_layer(inputs) #context length is 4 and batch size is 5
print(token_embeddings.shape)
print(token_embeddings)

torch.Size([5, 4, 768])
tensor([[[-0.3043, -0.7432, -0.5458,  ...,  0.8894,  0.2049,  0.9544],
         [ 0.2317,  0.0649,  0.0118,  ...,  0.7683,  0.0214,  0.9048],
         [ 0.5532,  0.2242,  0.6660,  ...,  1.2150,  0.8566,  0.6740],
         [-0.9550,  0.6920, -0.4026,  ..., -0.0889,  0.4213, -0.4788]],

        [[ 0.2317,  0.0649,  0.0118,  ...,  0.7683,  0.0214,  0.9048],
         [ 0.5532,  0.2242,  0.6660,  ...,  1.2150,  0.8566,  0.6740],
         [-0.9550,  0.6920, -0.4026,  ..., -0.0889,  0.4213, -0.4788],
         [ 0.5888, -0.2465, -0.6426,  ..., -0.5156, -0.7707, -0.2096]],

        [[ 0.5532,  0.2242,  0.6660,  ...,  1.2150,  0.8566,  0.6740],
         [-0.9550,  0.6920, -0.4026,  ..., -0.0889,  0.4213, -0.4788],
         [ 0.5888, -0.2465, -0.6426,  ..., -0.5156, -0.7707, -0.2096],
         [ 1.7876, -1.8311, -0.5405,  ..., -0.8679,  0.3276, -1.2629]],

        [[-0.9550,  0.6920, -0.4026,  ..., -0.0889,  0.4213, -0.4788],
         [ 0.5888, -0.2465, -0.6426,  ..., -0.5

In [45]:
pos_embeddings = pos_embedding_layer(torch.tensor([0,1,2,3])) #context length is 4
print(pos_embeddings.shape)
print(pos_embeddings)

torch.Size([4, 768])
tensor([[-0.2437,  0.9585, -0.9505,  ...,  0.2303, -0.2684,  0.7858],
        [-1.8610, -1.9265,  1.0855,  ...,  0.5151,  0.8646, -0.2780],
        [-0.2598,  0.0105,  0.0883,  ..., -2.1603,  0.3655,  1.3001],
        [ 0.7302,  0.2401, -0.9500,  ...,  0.9871, -0.2649,  0.1062]],
       grad_fn=<EmbeddingBackward0>)


In [46]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

torch.Size([5, 4, 768])
tensor([[[-5.4793e-01,  2.1527e-01, -1.4963e+00,  ...,  1.1197e+00,
          -6.3496e-02,  1.7402e+00],
         [-1.6292e+00, -1.8617e+00,  1.0974e+00,  ...,  1.2834e+00,
           8.8601e-01,  6.2678e-01],
         [ 2.9338e-01,  2.3473e-01,  7.5425e-01,  ..., -9.4526e-01,
           1.2222e+00,  1.9741e+00],
         [-2.2487e-01,  9.3212e-01, -1.3526e+00,  ...,  8.9826e-01,
           1.5645e-01, -3.7262e-01]],

        [[-1.1912e-02,  1.0233e+00, -9.3868e-01,  ...,  9.9862e-01,
          -2.4703e-01,  1.6906e+00],
         [-1.3078e+00, -1.7023e+00,  1.7515e+00,  ...,  1.7302e+00,
           1.7213e+00,  3.9600e-01],
         [-1.2149e+00,  7.0251e-01, -3.1437e-01,  ..., -2.2492e+00,
           7.8684e-01,  8.2130e-01],
         [ 1.3190e+00, -6.4301e-03, -1.5926e+00,  ...,  4.7150e-01,
          -1.0356e+00, -1.0339e-01]],

        [[ 3.0955e-01,  1.1827e+00, -2.8452e-01,  ...,  1.4454e+00,
           5.8823e-01,  1.4598e+00],
         [-2.8160e+00, -1.2